# LSTM MIMO + HHO — Sumber: Yahoo Finance

Model dioptimasi Harris Hawks Optimization untuk dataset **Yahoo Finance**. Pipeline preprocessing sama persis dengan notebook baseline sumber ini (`01_lstm_mimo_baseline_yfinance.ipynb`), memakai modul shared di `../src/`.

In [1]:
import sys, json, os, time, random, logging
sys.path.append('../src')

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping

from data_utils import prepare_source, inverse_close
from model_utils import build_lstm_mimo, decode_hyperparameters, DEFAULT_EPOCHS, DEFAULT_PATIENCE
from hho_utils import make_objective_function, run_hho_search
from evaluation import print_evaluation_report, summary_table

logging.getLogger('tensorflow').setLevel(logging.ERROR)
np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

## 1. Load & Preprocess

In [2]:
SOURCE = 'yfinance'
N_INPUT = 60      # window size (fixed, BUKAN dioptimasi HHO)
N_FORECAST = 5    # forecast horizon (fixed, BUKAN dioptimasi HHO)

ds = prepare_source(SOURCE, n_input=N_INPUT, n_forecast=N_FORECAST, data_dir='../data')

print(f"Sumber       : {ds['source']}")
print(f"Data shape   : {ds['data'].shape}")
print(f"Periode      : {ds['data'].index[0].date()} -> {ds['data'].index[-1].date()}")
print(f"X_train      : {ds['X_train'].shape}")
print(f"X_val        : {ds['X_val'].shape}")
print(f"X_test       : {ds['X_test'].shape}")

Sumber       : yfinance
Data shape   : (1336, 5)
Periode      : 2021-01-04 -> 2026-04-29
X_train      : (871, 60, 5)
X_val        : (135, 60, 5)
X_test       : (135, 60, 5)


## 2. Optimasi Hyperparameter dengan HHO

Ruang pencarian (6 dimensi): units layer 1 & 2, dropout layer 1 & 2, learning rate,
batch size. Window (n_input) dan horizon (n_forecast) **tidak** termasuk — keduanya
fixed di seluruh model (baseline maupun HHO) untuk menjaga fair comparison.

Epoch & patience dipakai penuh (100/10) baik saat pencarian kandidat maupun saat
training model final.

In [3]:
lb  = np.array([ 32,   32,  0.1,  0.1,  1e-4,  16])
ub  = np.array([256,  256,  0.5,  0.5,  1e-2, 128])
dim = 6

SEARCH_AGENTS_NO = 5
MAX_ITER         = 10
NUM_RUNS         = 3

objective_function = make_objective_function(
    ds['X_train'], ds['y_train'], ds['X_val'], ds['y_val'],
    n_input=N_INPUT, n_features=5, n_forecast=N_FORECAST
)

best_hp, best_fitness, best_solution, all_solutions = run_hho_search(
    objective_function, lb, ub, dim, SEARCH_AGENTS_NO, MAX_ITER, NUM_RUNS
)

units1, units2, drop1, drop2, lr, bs = decode_hyperparameters(best_hp)
print("\n" + "="*65)
print(f"  HYPERPARAMETER TERBAIK - YFINANCE")
print("="*65)
print(f"  units1={units1}  units2={units2}  dropout1={drop1:.4f}  "
      f"dropout2={drop2:.4f}  lr={lr:.6f}  batch_size={bs}")
print(f"  Best val_loss (fitness) : {best_fitness:.6f}")


Run 1/3
HHO mengoptimasi objective function (dim=6)


c:\Users\ASUS\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  Run 1 selesai dalam 8555.5s | val_loss=0.004499
  -> New best ditemukan di Run 1: val_loss=0.004499

Run 2/3
HHO mengoptimasi objective function (dim=6)
  Run 2 selesai dalam 5183.8s | val_loss=0.004893

Run 3/3
HHO mengoptimasi objective function (dim=6)


KeyboardInterrupt: 

## 3. Plot — HHO Convergence Curve

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(range(1, len(best_solution.convergence) + 1), best_solution.convergence,
         marker='o', markersize=4, lw=1.6)
plt.title('HHO Convergence Curve (Yahoo Finance)')
plt.xlabel('Iterasi'); plt.ylabel('Best val_loss (MSE)')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 4. Latih Model Final dengan Hyperparameter Terbaik (Epoch Penuh)

In [ ]:
model = build_lstm_mimo(N_INPUT, n_features=5, n_forecast=N_FORECAST,
                        units1=units1, units2=units2,
                        drop1=drop1, drop2=drop2, learning_rate=lr)
model.summary()

es = EarlyStopping(monitor='val_loss', patience=DEFAULT_PATIENCE,
                   restore_best_weights=True, verbose=0)

history = model.fit(
    ds['X_train'], ds['y_train'],
    validation_data=(ds['X_val'], ds['y_val']),
    epochs=DEFAULT_EPOCHS, batch_size=bs,
    shuffle=False, callbacks=[es], verbose=1
)
print(f"\nTraining selesai pada epoch ke-{len(history.history['loss'])}")

## 5. Prediksi & Evaluasi (RMSE, MAE, MAPE per Horizon)

In [ ]:
train_pred = inverse_close(ds['scaler'], model.predict(ds['X_train']))
val_pred   = inverse_close(ds['scaler'], model.predict(ds['X_val']))
test_pred  = inverse_close(ds['scaler'], model.predict(ds['X_test']))

splits = {
    'TRAIN': (ds['y_train_abs'], train_pred, ds['anchor_train']),
    'VAL':   (ds['y_val_abs'],   val_pred,   ds['anchor_val']),
    'TEST':  (ds['y_test_abs'],  test_pred,  ds['anchor_test']),
}

print_evaluation_report(f"LSTM MIMO + HHO - Yahoo Finance", N_FORECAST, splits)
summary_table(f"LSTM MIMO + HHO - Yahoo Finance", N_FORECAST, splits)

## 6. Plot — Training & Validation Loss (Model Final)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss', ls='--')
plt.title('LSTM MIMO + HHO (Yahoo Finance) - Training & Validation Loss')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Simpan Model & Metadata

In [ ]:
os.makedirs('../models', exist_ok=True)
model.save(f'../models/lstm_mimo_hho_yfinance.h5')

metadata = {
    "model_name": "LSTM MIMO + HHO",
    "source": "yfinance",
    "n_input": N_INPUT,
    "n_forecast": N_FORECAST,
    "hyperparameters": {
        "units_1": int(units1), "units_2": int(units2),
        "dropout_rate_1": float(drop1), "dropout_rate_2": float(drop2),
        "learning_rate": float(lr), "batch_size": int(bs)
    },
    "hho_search": {
        "search_agents_no": SEARCH_AGENTS_NO, "max_iter": MAX_ITER,
        "num_runs": NUM_RUNS, "best_fitness": float(best_fitness)
    },
    "normalization": "MinMaxScaler per-fitur (fit pada train only, per-sumber)",
    "split": "70/15/15 (walk-forward, time-ordered)"
}
with open(f'../models/lstm_mimo_hho_yfinance_meta.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Model & metadata HHO (Yahoo Finance) tersimpan di folder models/.")